In [1]:

from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd
from pandas import ExcelWriter
import datetime
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep
import os
import logging
import sys


print("Running US FCA Web Scraping Tool v.1.0")
regulatorName = "US FCA"
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__))
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

now=datetime.datetime.now()
filename= 'US FCA data {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

driver = webdriver.Chrome()
driver.maximize_window()


#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


url='https://apps.fca.gov/FCSPublicDirectory/PubSearchInstitution.aspx'
driver.get(url)
soup=BeautifulSoup(driver.page_source, 'html.parser')
soup=soup.find("div", {"id":"ctl00_cphMainContent_Update"})



clinks=[]
xname=[]

xfcanum=[]
xabname=[]
xstatus=[]
xphone=[]
xrssd=[]
xceo=[]
xcob=[]

xaddr=[]
xcity=[]
xcounty=[]
xzipcode=[]
xstate=[]
xweb=[]

xconame=[]
xchardate=[]
xcharnum=[]

entities=soup.find_all("tr")


process_date=now.strftime('%Y-%m-%d')

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}


	  

for entity in entities:
	ax = entity.find("a", href=True)
	xname.append(ax.text.strip())
	clinks.append('https://apps.fca.gov/FCSPublicDirectory/'+ax['href'])
clinks=clinks[1:]
xname=xname[1:]
for link in range(len(clinks)):
	print('Working on entity {} out of {}. (US FCA)'.format(link+1, len(clinks)))
	driver.get(clinks[link])
	soup2=BeautifulSoup(driver.page_source, 'html.parser')
	maintable=soup2.find("table", {"summary":"FCS Institution Directory"})
	maintable=maintable.find_all("tr")
	xfcanum.append(maintable[0].text[maintable[0].text.find(':')+1:].strip())
	xabname.append(maintable[1].text[maintable[1].text.find(':')+1:].strip())
	xstatus.append(maintable[2].text[maintable[2].text.find(':')+1:].strip())
	xphone.append(maintable[3].text[maintable[3].text.find(':')+1:].strip())
	xrssd.append(maintable[4].text[maintable[4].text.find(':')+1:].strip())
	xceo.append(maintable[5].text[maintable[5].text.find(':')+1:].strip())
	xcob.append(maintable[6].text[maintable[6].text.find(':')+1:].strip())

	addresstable=soup2.find("table",  {"summary":"Addresses"})
	county=addresstable.find_all("tr")[1].find_all('td')[1]
	website=addresstable.find_all("tr")[2].find_all('td')[1]
	addresstable=str(addresstable.find_all("tr")[0].find_all('td')[1]).replace('<br', '**<br').replace('</br', '**</br')
	addresstable=BeautifulSoup(addresstable, 'html.parser')
	line1=addresstable.text.split('**')[0].strip()
	line2=addresstable.text.split('**')[1].strip()
	zipcode=line2.split(',')[-1].strip()
	xaddr.append(line1)
	xcity.append(line2.split(',')[0].strip())
	xstate.append(zipcode.split(' ')[0])
	xzipcode.append(zipcode)
	xcounty.append(county.text.strip())
	xweb.append(website.text.strip())
	chartertable=soup2.find("table", {"summary":"Charter Information"})
	chartertable=chartertable.find_all("tr")
	if chartertable is not None:
		xconame.append(chartertable[0].text[chartertable[0].text.find(':')+1:].strip())
		xchardate.append(chartertable[1].text[chartertable[1].text.find(':')+1:].strip())
		xcharnum.append(chartertable[2].text[chartertable[2].text.find(':')+1:].strip())
	else:
		xconame.append('')
		xchardate.append('')
		xcharnum.append('')

df=pd.DataFrame({'Name': xname, 'FCA Institution Number': xfcanum,'Abbreviated Name': xabname,'Status': xstatus, 'Phone': xphone,
				 'RSSD Number': xrssd, 'CEO': xceo, 'Chairman of the Board': xcob, 'Address': xaddr, 'County': xcounty,
				 'City': xcity, 'State': xstate, 'Zip Code': xzipcode, 'Web URL': xweb, "Charter's Official Name": xconame, 
				 'Charter Date': xchardate,  'Charter Number': xcharnum})


# df.to_excel(writer, 'US FCA 1')

# writer.save()
# writer.close()

# sleep(3)

# driver.quit()

# endtime=datetime.datetime.now()
# difference=endtime-now
# difference=difference.total_seconds()
# file = open(filename.replace('data', 'time').replace('xlsx','txt'),'w') 
# file.write("Start: {} | End: {} | Total: {} hours and {} minutes".format(now, endtime, difference//3600, (difference%3600)/60))
# file.close() 
    
    
    

Running US FCA Web Scraping Tool v.1.0
Working on entity 1 out of 64. (US FCA)
Working on entity 2 out of 64. (US FCA)
Working on entity 3 out of 64. (US FCA)
Working on entity 4 out of 64. (US FCA)
Working on entity 5 out of 64. (US FCA)
Working on entity 6 out of 64. (US FCA)
Working on entity 7 out of 64. (US FCA)
Working on entity 8 out of 64. (US FCA)
Working on entity 9 out of 64. (US FCA)
Working on entity 10 out of 64. (US FCA)
Working on entity 11 out of 64. (US FCA)
Working on entity 12 out of 64. (US FCA)
Working on entity 13 out of 64. (US FCA)
Working on entity 14 out of 64. (US FCA)
Working on entity 15 out of 64. (US FCA)
Working on entity 16 out of 64. (US FCA)
Working on entity 17 out of 64. (US FCA)
Working on entity 18 out of 64. (US FCA)
Working on entity 19 out of 64. (US FCA)
Working on entity 20 out of 64. (US FCA)
Working on entity 21 out of 64. (US FCA)
Working on entity 22 out of 64. (US FCA)
Working on entity 23 out of 64. (US FCA)
Working on entity 24 out of

In [4]:
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}
for index, row in df.iterrows():
    name_  = row[0]
    fcanum_ = row[1]
    abname_ = row[2]
    status_ = row[3]
    phone_  = row[4]
    rssd_num_ = row[5]
    ceo_ = row[6]
    cob_ = row[7]
    addr_ = row[8]
    county_ = row[9]
    city_ = row[10]
    state_ = row[11]
    zipcode_ = row[12]
    web_ = row[13]
    coname_ = row[14]
    chardate_ = row[15]
    charnum_ = row[16]

    sqldict['Name'].append(name_)
    sqldict['InternalID_1'].append(fcanum_)
    sqldict['InternalID_1_type'].append('FCA Institution Number')
    sqldict['InternalID_2'].append(rssd_num_)
    sqldict['InternalID_2_type'].append('RSSD Number')

    sqldict['Phone'].append(phone_)
    sqldict['Address_1'].append(addr_)
    sqldict['City'].append(city_)
    sqldict['Cntry'].append(state_)
    sqldict['Zip'].append(zipcode_)
    sqldict['Website'].append(web_)
    sqldict['Name - Mother Company'].append(coname_)
    sqldict['ListProcessDate'].append(process_date)
    sqldict['RegCtry'].append('US')
    sqldict['RegCode'].append('FCA')
    sqldict['ListCode'].append('1')
    sqldict['RegulationType'].append('Regulated')
    sqldict['ListName'].append('List of "Farm Credit System Institutions"')
    sqldict = bourange_same_length_array(sqldict)
       


C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_25396\1377154530.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  name_  = row[0]
C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_25396\1377154530.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fcanum_ = row[1]
C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_25396\1377154530.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  abname_ = row[2]
C:\Users\wuj1\AppData\Local\Temp\4\ipykerne

In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df_total=pd.DataFrame(sqldict)
df_total.to_excel(filename, index=False)
# writer.save()
# writer.close()
driver.quit()
sleep(3)

AttributeError: 'XlsxWriter' object has no attribute 'save'